![image.png](https://i.imgur.com/a3uAqnb.png)

# How Language Models Behave: Families, Chat Templates and Decoding

- **Dataset**: [IMDb](https://huggingface.co/datasets/stanfordnlp/imdb) movie reviews from Hugging Face (25,000 test reviews labelled positive or negative) and the English-Arabic part of [OPUS-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100) (sentence pairs)
- **Task**: compare models from different families and stages on the same requests, use a pretrained model as a classifier, and see how chat templates and decoding settings change what a model writes
- **Model**: [GPT-2](https://huggingface.co/openai-community/gpt2) (124M, 2019), [Qwen3-0.6B-Base](https://huggingface.co/Qwen/Qwen3-0.6B-Base) (pretraining only) and [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) (the same model after post-training), used as they are; the tokenizers of five model families
- **Runtime**: Colab T4 GPU, about 10 minutes of compute

In this lab, we will:

- send the same requests to a 2019 model, a base model and a chat model, written plainly and written as text the web contains
- turn a pretrained model into a sentiment classifier with no training, zero-shot and with four examples
- print a chat template and see what a wrong template does
- reshape the next-token distribution with temperature and top-p, and measure how greedy decoding repeats itself
- count how tokenizers from different families split the same English and Arabic sentences

## A language model in brief

A decoder language model reads a sequence of tokens and outputs one score (logit) $z_i$ per vocabulary token for the next position. A softmax with temperature $T$ turns the scores into probabilities:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Where:
- $T = 1$ is the model's own distribution; $T < 1$ sharpens it and $T \to 0$ is greedy decoding (always the most probable token); $T > 1$ flattens it
- **top-$p$** keeps the smallest set of tokens whose probabilities add up to at least $p$, renormalises, and samples from it

A **base** model has only been pretrained to predict the next token on raw text, so it continues whatever it is given as if it were a document. A **chat** model is the same network after post-training on conversations, and expects its own **chat template**: special tokens that mark who is speaking.

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. Everything this lab needs ships with Colab.

In [ ]:
import collections
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast

import torch
import torch.nn.functional as F
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # hide informational warnings from model loading and generation

Every size and model name lives in the next cell. `N_REVIEWS` sets how many IMDb test reviews the classifier is scored on, `N_PAIRS` how many sentence pairs the tokenizers are measured on, and `N_STARTS` how many review openings the decoding comparison continues.

In [ ]:
SEED = 42
GPT2_ID = "openai-community/gpt2"
BASE_ID = "Qwen/Qwen3-0.6B-Base"
CHAT_ID = "Qwen/Qwen3-0.6B"
N_REVIEWS = 200            # IMDb test reviews for the classifier, half positive and half negative
REVIEW_WORDS = 200         # each review is cut to its first REVIEW_WORDS words
N_PAIRS = 500              # OPUS-100 English-Arabic pairs for the tokenizer comparison
N_STARTS = 10              # review openings continued by each decoding method
MAX_NEW_TOKENS = 40
CONTINUATION_TOKENS = 128  # length of the continuations in the decoding comparison

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so generation will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

## 1. Three models from two families

GPT-2 is OpenAI's 2019 decoder; its largest version showed that a language model can do some tasks with no fine-tuning when the task is written as text. Qwen3-0.6B-Base and Qwen3-0.6B share one architecture and differ only in training: the second was post-trained into an assistant. All three fit easily on a T4 together.

In [ ]:
def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE).to(DEVICE).eval()
    return tok, model

models = {name: load(model_id) for name, model_id in
          [("GPT-2", GPT2_ID), ("Qwen3-0.6B-Base", BASE_ID), ("Qwen3-0.6B (chat)", CHAT_ID)]}
for name, (tok, model) in models.items():
    print(f"{name:20s} {sum(p.numel() for p in model.parameters()) / 1e6:6.0f}M parameters, "
          f"vocabulary {len(tok):,}")

`continue_text` gives a model raw text and returns what it writes next, always taking the most probable token (greedy), so every run gives the same output.

In [ ]:
def continue_text(name, text, max_new_tokens=MAX_NEW_TOKENS, **decoding):
    tok, model = models[name]
    ids = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id,
                             do_sample=bool(decoding), **decoding)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

## 2. The same request, written two ways

First as a plain request, then as text that web pages already contain: a question followed by `A:`, or an English sentence followed by `Arabic:`. The chat model also gets the request through its chat template, the way it was trained to receive it.

In [ ]:
REQUESTS = {
    "plain question": "What is the capital of Saudi Arabia?",
    "question as a web pattern": "Q: What is the capital of Saudi Arabia?\nA:",
    "plain translation request": "Translate into Arabic: The library opens at nine in the morning.",
    "translation as a web pattern": "English: The library opens at nine in the morning.\nArabic:",
}
chat_tok = models["Qwen3-0.6B (chat)"][0]
def with_template(text):
    return chat_tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)

rows = []
for kind, text in REQUESTS.items():
    row = {"request": kind}
    for name in models:
        row[name] = continue_text(name, text).strip().replace("\n", " / ")[:120]
    rows.append(row)
rows.append({"request": "plain question, chat template",
             "Qwen3-0.6B (chat)": continue_text("Qwen3-0.6B (chat)", with_template(REQUESTS["plain question"])).strip()})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows).fillna("")

**What to notice.** A plain request is the start of a document for a base model: it may repeat it, add more questions or drift. Written as a pattern, the same request often gets the answer, followed by more of the pattern. Compare the 2019 model with the 2025 base model of a similar scale, and the chat model with and without its template.

## 3. A pretrained model as a classifier, with no training

Write the task as text and compare the probability of two possible next words. For a review followed by `Sentiment:`, the model's prediction is whichever of " positive" and " negative" it finds more probable. We try three prompts: the format alone, a one-line instruction before it, and the instruction plus four labelled short reviews from the training split.

In [ ]:
test = load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=SEED)
train = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=SEED)

def clean(text, n_words=None):
    words = text.replace("<br />", " ").split()
    return " ".join(words[:n_words] if n_words else words)

pos = [r for r in test.select(range(2000)) if r["label"] == 1][:N_REVIEWS // 2]
neg = [r for r in test.select(range(2000)) if r["label"] == 0][:N_REVIEWS // 2]
reviews = [(clean(r["text"], REVIEW_WORDS), r["label"]) for r in pos + neg]
short = [(clean(r["text"]), r["label"]) for r in train.select(range(3000)) if len(clean(r["text"]).split()) <= 60]
shots = [s for s in short if s[1] == 1][:2] + [s for s in short if s[1] == 0][:2]
random.Random(SEED).shuffle(shots)
NAME = {1: "positive", 0: "negative"}
HEAD = "Classify each movie review as positive or negative.\n\n"
PROMPTS = {
    "format only": lambda review: f"Review: {review}\nSentiment:",
    "+ instruction": lambda review: HEAD + f"Review: {review}\nSentiment:",
    "+ 4 examples": lambda review: HEAD + "".join(f"Review: {t}\nSentiment: {NAME[y]}\n\n" for t, y in shots)
                                   + f"Review: {review}\nSentiment:",
}
print(len(reviews), "test reviews;", "example:", PROMPTS["+ 4 examples"]("[the review]")[:400], "...")

`classify` runs one forward pass per review (no generation) and reads the next-token scores of " positive" and " negative". Prompts are padded on the right, so the last real token of each row is found from the attention mask.

In [ ]:
def classify(name, prompts, batch_size=8):
    tok, model = models[name]
    tok.padding_side = "right"
    pos_id = tok(" positive", add_special_tokens=False)["input_ids"][0]
    neg_id = tok(" negative", add_special_tokens=False)["input_ids"][0]
    preds = []
    for b in range(0, len(prompts), batch_size):
        enc = tok(prompts[b:b + batch_size], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        last = enc["attention_mask"].sum(1) - 1
        z = logits[torch.arange(len(last)), last]
        preds += (z[:, pos_id] > z[:, neg_id]).long().tolist()
    return preds

table = {}
for name in ["GPT-2", "Qwen3-0.6B-Base"]:
    t0 = time.time()
    table[name] = {}
    for kind, build in PROMPTS.items():
        preds = classify(name, [build(r) for r, _ in reviews])
        table[name][kind] = np.mean([p == y for p, (_, y) in zip(preds, reviews)]) * 100
    print(f"{name}: {time.time() - t0:.0f}s")
pd.DataFrame(table).T.round(1)

**What to notice.** Neither model was trained for sentiment, yet the format alone already sorts most reviews. Then compare the rows: does the instruction help, and do four examples help? A larger, more recent model makes much better use of instructions and examples in its input (in-context learning) than a small 2019 one.

## 4. Chat templates

A conversation is a list of messages; the tokenizer's chat template turns it into one string with special tokens marking each turn. Every model family has its own.

In [ ]:
conversation = [{"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Capital of Saudi Arabia?"},
                {"role": "assistant", "content": "Riyadh."},
                {"role": "user", "content": "And of Qatar?"}]
print(chat_tok.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False))

Now the same question to the chat model in three formats: plain text, the format another model family uses (`[INST] ... [/INST]`), and its own template.

In [ ]:
question = "Give me three tips for learning Arabic."
for label, text in [("plain text", question),
                    ("another family's format", f"[INST] {question} [/INST]"),
                    ("its own template", with_template(question))]:
    print(f"[{label}]\n{continue_text('Qwen3-0.6B (chat)', text, max_new_tokens=60).strip()}\n")

**What to notice.** Without its template the chat model behaves like a base model again: it continues the text, or imitates markup and keeps writing. Always build prompts with the tokenizer's own `apply_chat_template`; servers and libraries do this for you, and hand-built strings are where it goes wrong.

## 5. Decoding: from scores to words

The model gives a probability for every token in its vocabulary. The entropy $H = -\sum_i p_i \log_2 p_i$ measures, in bits, how spread out the distribution is. Compare a factual prompt with an open one.

In [ ]:
def next_token_probs(name, text, temperature=1.0):
    tok, model = models[name]
    ids = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    return F.softmax(logits / temperature, dim=-1)

def entropy_bits(p):
    p = p[p > 0]
    return float(-(p * p.log2()).sum())

base_tok = models["Qwen3-0.6B-Base"][0]
for text in ["The capital of Saudi Arabia is", "My favourite food is"]:
    p = next_token_probs("Qwen3-0.6B-Base", text)
    top = torch.topk(p, 5)
    print(f"{text!r}: entropy {entropy_bits(p):.1f} bits")
    for prob, idx in zip(top.values, top.indices):
        print(f"   {base_tok.decode([int(idx)])!r:14s} {prob:.3f}")

Temperature reshapes the distribution before sampling, and top-$p$ cuts its tail. For one open prompt, we print the most probable token's share, the entropy, and how many tokens top-$p = 0.9$ keeps, at three temperatures.

In [ ]:
PROMPT = "The best thing about living in Riyadh is"
for T in [0.5, 1.0, 1.5]:
    p = next_token_probs("Qwen3-0.6B-Base", PROMPT, temperature=T)
    sorted_p = torch.sort(p, descending=True).values
    kept = int((torch.cumsum(sorted_p, 0) < 0.9).sum()) + 1
    print(f"T = {T}: top token {sorted_p[0]:.2f}, entropy {entropy_bits(p):5.1f} bits, top-p 0.9 keeps {kept:,} tokens")

for T in [0.2, 0.7, 1.5]:
    torch.manual_seed(SEED)
    samples = [continue_text("Qwen3-0.6B-Base", PROMPT, max_new_tokens=20, temperature=T, top_k=0, top_p=1.0)
               for _ in range(2)]
    print(f"\nT = {T}:", *[f"\n   ...{s.strip()}" for s in samples])

**What to notice.** At low temperature the samples are safe and alike; at high temperature thousands of unlikely tokens share the probability and the text drifts off topic or breaks. Top-$p$ adapts to the prompt: it keeps few tokens when the model is confident and many when it is not.

## 6. The most probable text repeats itself

Greedy decoding and beam search pick the most probable continuation. On open-ended text that tends to loop. We continue the first 20 words of `N_STARTS` reviews for `CONTINUATION_TOKENS` tokens with four methods and measure the share of distinct 4-token sequences (1.0 means no 4-gram repeats).

In [ ]:
starts = [" ".join(clean(r["text"]).split()[:20]) for r in test.select(range(2000, 2000 + N_STARTS))]
METHODS = {"greedy": dict(do_sample=False),
           "beam search, 4 beams": dict(do_sample=False, num_beams=4),
           "sample, T 0.7, top-p 0.8": dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=0),
           "sample, T 1.0, top-p 0.95": dict(do_sample=True, temperature=1.0, top_p=0.95, top_k=0)}

def distinct_4grams(ids):
    grams = [tuple(ids[i:i + 4]) for i in range(len(ids) - 3)]
    return len(set(grams)) / max(1, len(grams))

tok, model = models["Qwen3-0.6B-Base"]
scores, examples = {}, {}
for method, kwargs in METHODS.items():
    torch.manual_seed(SEED)
    values = []
    for s in starts:
        ids = tok(s, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=CONTINUATION_TOKENS, min_new_tokens=CONTINUATION_TOKENS,
                                 pad_token_id=tok.pad_token_id, **kwargs)
        new = out[0, ids["input_ids"].shape[1]:].tolist()
        values.append(distinct_4grams(new))
    scores[method] = np.mean(values)
    examples[method] = tok.decode(new, skip_special_tokens=True)[:200]

plt.figure(figsize=(7, 3))
plt.barh(list(scores), list(scores.values()))
plt.xlabel("distinct 4-grams (higher = less repetition)")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()
for method, text in examples.items():
    print(f"[{method}] ...{text}\n")

**What to notice.** Greedy and beam search fall into loops; sampling with a moderate temperature and top-$p$ stays varied. Use greedy decoding when there is one right answer (a label, an extraction), and the sampling settings on the model card for open text.

## 7. Tokenizers across model families

Every family trains its own tokenizer, and the training text decides how well it splits each language. We count tokens per word on the same English and Arabic sentences for five families, including two built for Arabic (ALLaM and Jais). Only the tokenizers are downloaded, not the models.

In [ ]:
pairs = load_dataset("Helsinki-NLP/opus-100", "ar-en", split="test")
pairs = [(r["translation"]["en"], r["translation"]["ar"]) for r in pairs]
pairs = [(e, a) for e, a in pairs if 8 <= len(e.split()) <= 40][:N_PAIRS]
words_en = sum(len(e.split()) for e, _ in pairs)
words_ar = sum(len(a.split()) for _, a in pairs)

TOKENIZERS = {"GPT-2": GPT2_ID, "Qwen3": CHAT_ID, "Llama 3.2": "unsloth/Llama-3.2-1B-Instruct",
              "ALLaM": "humain-ai/ALLaM-7B-Instruct-preview", "Jais": "inception42/jais-family-590m-chat"}
rows = []
for family, model_id in TOKENIZERS.items():
    # Jais maps its tokenizer through custom code; its tokenizer.json loads as a plain fast tokenizer
    t = PreTrainedTokenizerFast.from_pretrained(model_id) if "jais" in model_id else AutoTokenizer.from_pretrained(model_id)
    n_en = sum(len(t(e, add_special_tokens=False)["input_ids"]) for e, _ in pairs)
    n_ar = sum(len(t(a, add_special_tokens=False)["input_ids"]) for _, a in pairs)
    rows.append({"family": family, "vocabulary": len(t), "English tokens per word": n_en / words_en,
                 "Arabic tokens per word": n_ar / words_ar, "Arabic / English tokens": n_ar / n_en})
pd.DataFrame(rows).set_index("family").round(2)

**What to notice.** The same Arabic text can cost several times more tokens in one family than in another. More tokens means less text in the context window, slower generation and a higher bill per document: one more reason to compare families on your own language before choosing a model.

## Summary

- A base model continues text; written as a familiar pattern, a request often gets an answer, and a chat model answers plain requests only through its own template.
- A pretrained model can classify with no training by comparing next-token probabilities, and larger, newer models use instructions and examples better.
- Temperature and top-$p$ trade variety against coherence; greedy decoding and beam search loop on open-ended text.
- Tokenizers from different families split the same Arabic text into very different numbers of tokens.

## Exercises

1. Change `BASE_ID` to `"Qwen/Qwen3-1.7B-Base"` and rerun section 3. How much do the instruction and the examples help the larger model?
2. In section 2, write a few-shot pattern for translation (two English-Arabic pairs, then the request). Does GPT-2 translate?
3. In section 5, try `PROMPT = "The capital of Saudi Arabia is"` at the three temperatures. Why does temperature matter less here?
4. Add `repetition_penalty=1.3` to the greedy method in section 6. Does it remove the loops, and what does the text look like?
5. Add the tokenizer of another Arabic-centric family, `"QCRI/Fanar-1-9B-Instruct"`, to section 7. Where does it fall between Qwen3 and Jais?

## Further Reading

- [Radford et al., 2019. Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- [Brown et al., 2020. Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165)
- [Holtzman et al., 2020. The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751)
- [Qwen Team, 2025. Qwen3 Technical Report](https://arxiv.org/abs/2505.09388)
- [Petrov et al., 2023. Language Model Tokenizers Introduce Unfairness Between Languages](https://arxiv.org/abs/2305.15425)

### Contributed by: Yazen Shaebi